# BERT for Quora Question Pairs

In [1]:
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torchmetrics.functional.classification import (
    binary_accuracy,
    binary_precision,
    binary_recall,
    binary_f1_score,
)
from transformers import AutoTokenizer, AutoModel, DataCollatorWithPadding
from datasets import load_dataset
import lightning as L
from lightning.pytorch.callbacks.early_stopping import EarlyStopping

In [2]:
DATASET_NAME = "AlekseyKorshuk/quora-question-pairs"
MODEL_NAME = "google-bert/bert-base-uncased"
SEED = 42
BATCH_SIZE = 32
NUM_WORKERS = 4
D_MODEL = 768
LEARNING_RATE = 3e-5
MAX_EPOCHS = 2
PATIENCE = 3

## 1. Prepare dataset

In [3]:
ds = load_dataset(DATASET_NAME, split="train")

In [4]:
ds

Dataset({
    features: ['id', 'qid1', 'qid2', 'question1', 'question2', 'is_duplicate'],
    num_rows: 404290
})

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [6]:
ds_cleaned = ds.filter(
    lambda row: isinstance(row["question1"], str) and isinstance(row["question2"], str)
)
ds_tokenized = ds_cleaned.map(
    lambda row: tokenizer(row["question1"], row["question2"], truncation=True),
    batched=True,
    remove_columns=["qid1", "qid2", "question1", "question2"],
)

In [7]:
ds_split_1 = ds_tokenized.train_test_split(0.3)
ds_split_2 = ds_split_1["test"].train_test_split(0.5)
ds_train, ds_val, ds_test = ds_split_1["train"], ds_split_2["train"], ds_split_2["test"]

In [8]:
ds_train, ds_val, ds_test

(Dataset({
     features: ['id', 'is_duplicate', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 283000
 }),
 Dataset({
     features: ['id', 'is_duplicate', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 60643
 }),
 Dataset({
     features: ['id', 'is_duplicate', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 60644
 }))

In [9]:
data_collator = DataCollatorWithPadding(tokenizer)

In [10]:
train_loader = DataLoader(
    ds_train,
    BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=data_collator,
)
val_loader = DataLoader(
    ds_val,
    BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=data_collator,
)
test_loader = DataLoader(
    ds_test,
    BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=data_collator,
)

## 2. Prepare model

In [11]:
class BertQuoraModel(L.LightningModule):
    def __init__(self, bert, d_model=D_MODEL):
        super().__init__()
        self.bert = bert
        self.d_model = d_model
        self.classifier = nn.Linear(d_model, 1)

    def forward(self, input_ids, attention_mask, token_type_ids):
        bert_output = self.bert(input_ids, attention_mask, token_type_ids)
        cls_token = bert_output.pooler_output
        logits = self.classifier(cls_token).reshape(-1)
        return logits

    def training_step(self, batch, batch_idx):
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        token_type_ids = batch["token_type_ids"]
        labels = batch["is_duplicate"].float()

        logits = self(input_ids, attention_mask, token_type_ids)
        loss = F.binary_cross_entropy_with_logits(logits, labels)
        preds = logits > 0

        accuracy = binary_accuracy(preds, labels)
        precision = binary_precision(preds, labels)
        recall = binary_recall(preds, labels)
        f1 = binary_f1_score(preds, labels)

        self.log("train_loss", loss)
        self.log("train_accuracy", accuracy)
        self.log("train_precision", precision)
        self.log("train_recall", recall)
        self.log("train_f1", f1)

        return loss

    def validation_step(self, batch, batch_idx):
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        token_type_ids = batch["token_type_ids"]
        labels = batch["is_duplicate"].float()

        logits = self(input_ids, attention_mask, token_type_ids)
        loss = F.binary_cross_entropy_with_logits(logits, labels)
        preds = logits > 0

        accuracy = binary_accuracy(preds, labels)
        precision = binary_precision(preds, labels)
        recall = binary_recall(preds, labels)
        f1 = binary_f1_score(preds, labels)

        self.log("val_loss", loss)
        self.log("val_accuracy", accuracy)
        self.log("val_precision", precision)
        self.log("val_recall", recall)
        self.log("val_f1", f1)

    def test_step(self, batch, batch_idx):
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        token_type_ids = batch["token_type_ids"]
        labels = batch["is_duplicate"].float()

        logits = self(input_ids, attention_mask, token_type_ids)
        loss = F.binary_cross_entropy_with_logits(logits, labels)
        preds = logits > 0

        accuracy = binary_accuracy(preds, labels)
        precision = binary_precision(preds, labels)
        recall = binary_recall(preds, labels)
        f1 = binary_f1_score(preds, labels)

        self.log("test_loss", loss)
        self.log("test_accuracy", accuracy)
        self.log("test_precision", precision)
        self.log("test_recall", recall)
        self.log("test_f1", f1)

    def configure_optimizers(self):
        optimizer = AdamW(self.parameters(), LEARNING_RATE)
        return optimizer

## 3. Train model

In [12]:
bert = AutoModel.from_pretrained(MODEL_NAME)
bert.train()
model = BertQuoraModel(bert)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
trainer = L.Trainer(
    callbacks=[EarlyStopping("val_loss", patience=PATIENCE)],
    max_epochs=MAX_EPOCHS
)
trainer.fit(model, train_loader, val_loader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
You are using a CUDA device ('NVIDIA GeForce RTX 4060 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ bert       │ BertModel │  109 M │ train │     0 │
│ 1 │ classifier │ Linear    │    769 │ train │     0 │
└───┴────────────┴───────────┴────────┴───────┴───────┘

Trainable params: 109 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 109 M                                                                                                
Total estimated model params size (MB): 437.932                                                                    
Modules in train mode: 229                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/home/minht/miniconda3/envs/nlp2/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

In [ ]:
trainer.test(dataloaders=test_loader)

TypeError: `Trainer.test()` requires a `LightningModule` when it hasn't been passed in a previous run